# 06: Evaluate all methods

Evaluates the four methods on the 1,000 test rows in one session, on one GPU, with the
same settings:

- **Short answer:** yes/no from the model's next-token probabilities after
  "Simplified Answer:" (deterministic, no LLM judge). Also *written accuracy*: how often
  the yes/no the model writes in its answer matches the reference.
- **Long answer:** greedy decoding. The explanation after "Detailed Answer" is scored
  with NLI (reference as premise, answer as hypothesis), METEOR and BERTScore; plus the
  share of answers in the required format.
- **95% bootstrap confidence intervals** over the test rows, and paired McNemar tests
  of each method's accuracy against the baseline.

**Runtime:** Colab GPU (A100). About 30-40 minutes.

**Needs:**
- the `HF_TOKEN` Colab secret, and the splits from `02_prepare_data`;
- `runs/v2/validation_choices.json` from notebook 03;
- the adapters listed in `ADAPTERS` below (from notebooks 04 and 05).

**Outputs** go to `MyDrive/biomedical-qa-peft/runs/v2/<method>/` (`predictions.jsonl`,
`results.json`). If Colab disconnects, rerunning reuses saved predictions; predictions
made with different settings raise an error instead (delete that folder).

In [ ]:
# Colab bootstrap: fetch the latest code from GitHub. Does nothing locally.
# Same cell in every notebook. To pick up newly pushed code in Colab:
# Runtime > Restart session, then run all cells.
import sys

REPO = "dgelbier23/biomedical-qa-peft"

if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path

    repo_dir = Path("/content") / REPO.split("/")[1]
    subprocess.run(["rm", "-rf", str(repo_dir)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git",
                    str(repo_dir)], check=True)
    sys.path.insert(0, str(repo_dir / "src"))

In [ ]:
# Memory and tokenizer settings from the original; set before CUDA starts.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Colab: install the pinned library versions. Locally, use requirements.txt.
import subprocess
import sys

from biomedical_qa_peft.runtime.env import REPO_ROOT, in_colab

if in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_ROOT / "requirements-colab.txt")], check=True)

## Setup

In [ ]:
import json

import pandas as pd

from biomedical_qa_peft.evaluation.config import BASELINE_PROMPT, CANDIDATE_PROMPTS, InferenceConfig
from biomedical_qa_peft.evaluation.inference import load_base_model
from biomedical_qa_peft.evaluation.scoring import load_scorers
from biomedical_qa_peft.runtime.env import describe_environment, get_output_dir

for key, value in describe_environment().items():
    print(f"{key:>10}: {value}")

runs_dir = get_output_dir("runs")
v2_dir = runs_dir / "v2"
choices = json.loads((v2_dir / "validation_choices.json").read_text())
candidates = {f"candidate_{i}": p for i, p in enumerate(CANDIDATE_PROMPTS, start=1)}
config = InferenceConfig()
system_prompts = {
    "baseline": BASELINE_PROMPT,
    "prompt_engineering": candidates[choices["prompt_engineering_prompt"]],
    "prefix_tuning": BASELINE_PROMPT,
    "lora": BASELINE_PROMPT,
}
# The adapters trained by notebooks 04 and 05.
ADAPTERS = {
    "prefix_tuning": v2_dir / "train" / "prefix_tuning" / "adapter",
    "lora": v2_dir / "train" / "lora" / "adapter",
}

model, tokenizer = load_base_model(config)
scorers = load_scorers(config)
test = pd.read_json(get_output_dir("data") / "pubmedqa" / "test.jsonl", lines=True)
formatted_texts = test["formatted_text"].tolist()
print(f"{len(formatted_texts)} test rows; prompt engineering uses "
      f"{choices['prompt_engineering_prompt']}")

## 1. Evaluate the four methods

In [ ]:
from peft import PeftModel

from biomedical_qa_peft.evaluation.scoring import run_method

for method, system_prompt in system_prompts.items():
    evaluated = PeftModel.from_pretrained(model, str(ADAPTERS[method])) if method in ADAPTERS else model
    run_method(v2_dir / method, evaluated, tokenizer, formatted_texts, system_prompt, scorers,
               config, method=method, adapter=str(ADAPTERS.get(method)),
               validation_choices=choices)
    if method == "lora":
        evaluated.unload()  # LoRA changes `model` in place; restore the base model

## 2. Are the differences real?

Paired exact McNemar tests on the same 1,000 rows: each method's yes/no accuracy against
the baseline's. p < 0.05 means the two methods' error rates differ by more than chance.

In [ ]:
from biomedical_qa_peft.evaluation.metrics import mcnemar_p


def correct(method: str) -> list[bool]:
    short = json.loads((v2_dir / method / "results.json").read_text())["results"]["short_answer"]
    return [p == r for p, r in zip(short["predictions"], short["references"])]


baseline_correct = correct("baseline")
pd.Series({method: mcnemar_p(correct(method), baseline_correct)
           for method in system_prompts if method != "baseline"},
          name="McNemar p vs baseline").round(4)

## 3. Results with 95% confidence intervals

Bootstrap intervals over the test rows, for the metrics that have one.

In [ ]:
table = {}
for method in system_prompts:
    record = json.loads((v2_dir / method / "results.json").read_text())
    for metric, value in record["summary"].items():
        ci = record["confidence_intervals"].get(metric)
        table[(method, metric)] = {
            "value": round(value, 3),
            "95% CI": f"[{ci[0]:.3g}, {ci[1]:.3g}]" if ci else "",
        }
pd.DataFrame(table).T